# 03 · 多列指标 & 能力探测

本库不止有两列变量的指标（如 销售额=单价×销量），还专门放了**需要 3 个/4 个字段同时参与**才算得出来的派生指标：

- 单均净利润 = 实收金额 − 平台抽成 − 成本 （3 列）
- 综合成本率 = (平台抽成 + 成本 + 税费) / 实收金额 （4 列）
- 销售净利率 = (销售额 − 成本 − 平台扣点 − 推广费) / 销售额 （4 列）
- 互动率 = (点赞 + 收藏 + 评论) / 阅读播放 （4 列）
- 综合履约分 = 准时×0.4 + 好评×0.4 + (1−纠纷)×0.1 + 复购×0.1 （4 列）

再配合 **能力探测 / 一键计算**，给一份数据就自动算，不用背每个指标要哪几列。

In [ ]:
import da, pandas as pd

sk = pd.DataFrame({
    "实收金额": [1000, 2000, 1500],
    "平台抽成": [100, 200, 150],
    "成本": [200, 400, 300],
    "税费": [50, 100, 75],
    "报价金额": [1200, 2400, 1800],
    "交付时长h": [10, 20, 15],
    "渠道来源": ["甲", "乙", "甲"],
})
sk

In [ ]:
# 3 列 / 4 列指标，直接调
da.技能外包.单均净利润(sk)
da.技能外包.综合成本率(sk)
da.技能外包.时薪净利(sk)
da.技能外包.渠道净收益率(sk)   # 按 渠道来源 分组

## 能力探测：给份数据，自动告诉你能算哪些

只差 1 个字段就能解锁的指标也会提示你——这就是「把三列四列同时考虑，尽量榨出可得数据」。

In [ ]:
partial = pd.DataFrame({
    "销售额": [1000, 2000],
    "成本": [400, 800],
    "平台扣点": [50, 100],   # 缺 推广费 → 销售净利率 差1列可解锁
})
da.能力探测(partial, "小微电商")

In [ ]:
# 一键把所有能算的指标算出来
res = da.一键计算(partial)
pd.Series(res).sort_values(ascending=False)

## 图表参数注释速记

每个图表函数都带详细参数 docstring，`da.图表.柱状图??` 或 `da.图表.查看配方(da.图表.柱状图)` 可看。关键点：

- `agg`：同类怎么汇总（sum/mean/count/None）。
- `top_n`：只画前 N 类，应对长尾。
- `预设`：换**黑底**用 `预设="暗夜"`（清爽白/商务蓝/活力橙/暗夜）。

In [ ]:
df = pd.read_csv("data/sample_region_sales.csv")
da.图表.柱状图(df, x="地区", y="销售额", 预设="暗夜")   # 黑底
da.图表.柱状图(df, x="地区", y="销售额", agg="sum", top_n=5)  # 只画前5

## 指标缺列容错：数据少几列也能算

有些**附加成本/费用类**字段（税费、平台抽成、成本…）没记也没关系，标记为「可选列」后缺了自动按 0 算（视为"没发生"）。
分母类列（实收金额 / 销售额 …）才必须齐全，缺了就是真算不了。

`da.能力探测` 会在 ✅ 列表里用 ⚠️ 标出"哪些可选列被按 0 算了"。

In [ ]:
sk2 = sk.drop(columns=["税费"])            # 故意删掉"税费"
da.技能外包.综合成本率(sk2)              # 税费按 0 算，照常出数，不崩
da.能力探测(sk2, "技能外包")            # ✅ 会标 ⚠️ 可选列['税费'] 缺失 → 按0算